In [1]:
import os
import pynwb
import h5py
from tqdm import tqdm
import pandas as pd
from pathlib import Path
import numpy as np
import seaborn as sns
import pywt
import matplotlib
import graph
import time

from joblib import Parallel, delayed

import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib import gridspec

import LFP
import load_data

In [2]:
def process_channel(ch, lfpData, band):
    
    preprocessed = LFP.preprocess_LFP(lfpData[:, ch])
    band_sig, power, phase = LFP.freq_band(preprocessed, band)
    
    return band_sig, power, phase

In [10]:
band = [13, 30]

for date in ['032023','032823','033123','040423','040723']:
    
    timestamps, lfpData = load_data.load_LFP(date)
    lfpData = np.asarray(lfpData)
    nchannels = lfpData.shape[1]
    
    # parallel over channels
    with Parallel(n_jobs=-1, backend="threading") as parallel:
        results = parallel(
            delayed(process_channel)(ch, lfpData, band)
            for ch in tqdm(range(nchannels), desc=f"Channels {date}")
        )
    
    # unpack results
    band_sig = np.column_stack([r[0] for r in results])
    power    = np.column_stack([r[1] for r in results])
    phase    = np.column_stack([r[2] for r in results])
    
    # save
    
    with h5py.File(f"D:/Alpha/Session_{date}_{band[0]}_{band[1]}.h5", "w") as f:
        f.create_dataset("band_sig", data=band_sig, dtype="float32")
        f.create_dataset("power", data=power, dtype="float32")
        f.create_dataset("phase", data=phase, dtype="float32")



Channels 032023: 100%|██████████| 160/160 [01:32<00:00,  1.74it/s]

Channels 032823: 100%|██████████| 192/192 [03:06<00:00,  1.03it/s]

Channels 033123: 100%|██████████| 192/192 [03:22<00:00,  1.06s/it]

Channels 040423:  88%|████████▊ | 168/192 [03:09<00:23,  1.00it/s]

MemoryError: bad allocation